<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex09.1-burgers-2d/Ex09.1_03_reynolds_sweep.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_09.1 · Notebook 03 — the Reynolds sweep

**Paired with L9.1 · Laminar Flow**

The point of this notebook is to find the limit of the method **for yourself**,
rather than being told where it is.

As Re rises the front steepens, the convection term dominates, and at some
point a smooth network can no longer follow the solution. Find that point.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.1-burgers-2d/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
# --- paste your residual_fn and loss_fn_factory from notebook 01 here ---
raise NotImplementedError("paste your residual_fn and loss_fn_factory")

## 1 · Sweep Re and record the error

Seven runs. Each one is a full training, so start it and leave it — and time a
single run at the default Re before you commit to the whole sweep.

In [ ]:
sweep = []
for Re in (5, 10, 20, 50, 100, 200, 500):
    cfg = pb.Config(reynolds=Re, n_collocation=6000, n_hidden=40, n_layers=5,
                    adam_epochs=2000, lbfgs_epochs=200)
    sweep.append(pb.run_study(cfg, residual_fn, loss_fn_factory, verbose=False))
    r = sweep[-1]
    print(f"Re = {Re:<5} nu {cfg.nu:.4g}   front width {8*cfg.nu:.3g}   "
          f"loss {r['final_loss']:.2e}   mean rel L2 (u) {r['mean_u_rel']:.3e}")

In [ ]:
Res  = [r["config"].reynolds for r in sweep]
errs = [r["mean_u_rel"] for r in sweep]
plt.figure(figsize=(6.5, 3.4))
plt.loglog(Res, errs, "o-", color="#d94f2b")
plt.xlabel("Reynolds number"); plt.ylabel("mean relative $L_2$ error in u")
plt.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()

print(error_table(
    [[f"{r['config'].reynolds:g}", f"{r['config'].nu:.4g}",
      f"{8*r['config'].nu:.3g}", f"{r['final_loss']:.2e}",
      f"{r['mean_u_rel']:.3e}", f"{r['mean_v_rel']:.3e}"] for r in sweep],
    ["Re", "nu", "front width", "final loss", "mean rel L2 (u)", "mean rel L2 (v)"]))

## 2 · The question this notebook exists to ask

Look at the highest-Re run that "converged". Plot its field against the exact
solution and inspect the front.

A PINN that fails at high Re usually does **not** produce an obviously broken
result. It produces a smooth, plausible field that is simply wrong — the front
is smeared rather than sharp. That is the failure mode described on slide 22,
and it is dangerous precisely because it looks reasonable.

In [ ]:
worst = sweep[-1]
pb.plot_fields(worst, t=0.5)

X, Y, U, V, Ue, Ve = pb.eval_field(worst["model"], 0.5, worst["config"].nu)
d = np.linspace(0, 1, U.shape[0])
plt.figure(figsize=(6.5, 3.2))
plt.plot(d, np.diag(Ue), label="exact", color="#1f77b4")
plt.plot(d, np.diag(U), "--", label="PINN", color="#d94f2b")
plt.title(f"front profile, Re = {worst['config'].reynolds:g}")
plt.legend(); plt.tight_layout(); plt.show()

**Read the loss and the error side by side.** A run whose loss fell as
far as the low-Re runs but whose error is an order of magnitude worse has
satisfied the equations at the points you gave it and done something else
between them. The collocation set cannot resolve a front $8\nu$ wide once
$8\nu$ is smaller than the spacing between neighbouring points, and no amount
of further optimisation will tell you so.

## 3 · Save

In [ ]:
import pickle

os.makedirs(pb.OUTPUT_DIR, exist_ok=True)
path = os.path.join(pb.OUTPUT_DIR, "nb03_sweep.pkl")
with open(path, "wb") as f:
    pickle.dump([{k: v for k, v in r.items() if k != "model"} for r in sweep], f)
print("wrote", path)